<a href="https://colab.research.google.com/github/Flaviomental1/Modelizado-de-Sistemas-con-AI/blob/main/Copia_de_Practica_Unidad_1_Sietemas_Expertos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desafío Individual: Sistema de Gestión de Obra Inteligente

## Contexto del Problema
Una empresa constructora está desarrollando una torre de gran altura y necesita automatizar dos procesos críticos para garantizar la seguridad y la eficiencia operativa:
* Evaluación de Riesgos en Obra (Lógica Deductiva): Determinar si es seguro continuar con las tareas de altura o excavación basándose en sensores climáticos y estructurales.
* Planificación de Maquinaria Pesada (Satisfacción de Restricciones): Asignar equipos (grúas, excavadoras) a zonas específicas del predio respetando límites de seguridad y espacio físico.

**Misión A:** Diagnóstico de Seguridad con expertaDebes programar un motor de inferencia que reciba datos de sensores y devuelva el nivel de riesgo del sitio. Este sistema actúa como un Cerebro Lógico para evitar accidentes.

Reglas a Implementar:
* **Riesgo Crítico** (Paro de Obra):  Si la velocidad del viento es $> 60\ km/h$ o si se detectan grietas en el suelo de fundación.
* **Riesgo Moderado** (Precaución): Si la velocidad del viento está entre $40\ km/h$ y $60\ km/h$ o si hay humedad extrema en zonas de excavación.
* **Bajo Riesgo** (Operación Normal): Si los vientos son $< 40\ km/h$ y no hay alertas estructurales activas.

**Consigna Técnica:** Utiliza el parámetro salience para asegurar que la regla de Riesgo Crítico se evalúe con la máxima prioridad ante cualquier otra condición.El sistema debe imprimir el diagnóstico final y la orden de seguridad correspondiente.

### Mision A

In [ ]:
# Instalación
!pip install experta
!pip install --upgrade frozendict

  Preparing metadata (setup.py) ... done
  Created wheel for frozendict: filename=frozendict-1.2-py3-none-any.whl size=3149 sha256=a9b7dab355032605cdb568d4aa634f42df564c821ce035fd38b6d6165709a10a
  Stored in directory: /root/.cache/pip/wheels/f6/ff/aa/750fec7bf9618d87b53572def5abf3e098f853cc5ab4147656
Successfully built frozendict
  Attempting uninstall: frozendict
    Found existing installation: frozendict 2.4.7
    Uninstalling frozendict-2.4.7:
      Successfully uninstalled frozendict-2.4.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.66 requires frozendict>=2.3.4, but you have frozendict 1.2 which is incompatible.
  Attempting uninstall: frozendict
    Found existing installation: frozendict 1.2
    Uninstalling frozendict-1.2:
      Successfully uninstalled frozendict-1.2
ERROR: pip's dependency resolver does not currently take into accou

In [ ]:
# @title
from typing import Self
from experta import *

class EstadoObra(Fact):
    """Datos capturados por sensores en el sitio"""
    pass
class MotorSeguridad(KnowledgeEngine):
    # TODO: Implementar las reglas de seguridad aquí.

    #indicamos que declarar si hay vientos mayores a 60 KM/H y grietas.
    # Agregamos un "salience para riesgo critico"
    @Rule(EstadoObra(viento=(lambda v: v > 60))|(EstadoObra(grietas=True,humedad=True)), salience=100)
    def riesgo_critico(self):
        print("Riesgo Crítico: Paro de Obra")
        self.declare(EstadoObra(riesgo="Riesgo Crítico"))


    # Indicamos que declarar si hay vientos mayores a 60 M/h y  hay humedad

    @Rule(EstadoObra(viento=(lambda v: 40 < v <60))|(EstadoObra(humedad=False,grietas= True)))
    def riesgo_moderado(self):
        print("Riesgo Moderado: Precaución")
        self.declare(EstadoObra(riesgo="Riesgo Moderado"))

    # Indicamos que declarar si hay vientos menores a 40 M/h, no hay grietas y no hay humedad.

    @Rule(EstadoObra(viento=(lambda v: v < 40))|(EstadoObra(grietas=False, humedad=False)))
    def bajo_riesgo(self):
        print("Bajo Riesgo: Operación Normal")
        self.declare(EstadoObra(riesgo="Bajo Riesgo"))



#Ejecucion
engine = MotorSeguridad()
engine.reset()
engine.declare(EstadoObra(viento=65,humedad=True,  grietas=True))
engine.run()
pass

# Prueba el sistema con viento de 65 km/h y presencia de grietas

Riesgo Crítico: Paro de Obra


### Mision B

**Misión B:** Ubicación de Equipos con python-constraint

Debes encontrar la distribución óptima de 3 máquinas pesadas en 3 zonas de trabajo distintas. El sistema debe "podar" las opciones que violen las normativas de seguridad.

Restricciones (Reglas de Oro):
* **Grúa Torre:** Solo puede ubicarse en la Zona_Estable (debido a la necesidad de una base de hormigón reforzada).
* **Excavadora:** No puede ingresar a la Zona_Estrecha debido a sus dimensiones.
* **Hormigonera:** No puede estar en la misma zona que la Grúa Torre para evitar congestión de camiones.
* **Exclusividad:** Cada zona solo puede albergar una máquina a la vez para evitar colisiones.Consigna Técnica:Define las variables (Máquinas) y el dominio (Zonas de la obra).

Aplica las funciones de restricción para que el motor de búsqueda encuentre la única configuración válida.

# Mision B

In [ ]:
!pip install python-constraint

  Preparing metadata (setup.py) ... done
  Created wheel for python-constraint: filename=python_constraint-1.4.0-py2.py3-none-any.whl size=24061 sha256=36d07477e139862af95585004eaf8b80814ebc1d1d6c5f19a3ebebb4757d7c85
  Stored in directory: /root/.cache/pip/wheels/c1/d2/3d/082849b61a9c6de02d4a7c8a402c224640f08d8a971307b92b
Successfully built python-constraint


In [ ]:
from constraint import *

def planificar_maquinaria():
    problem = Problem()

    # Variables (Máquinas)
    maquinas = ["GrúaTorre", "Excavadora", "Hormigonera"]
    # Dominio (Zonas de la obra)
    zonas = ["zona_Estable", "zona_Estrecha", "zona_Espera"]

    # Definimos las variables y dominios
    problem.addVariables(maquinas, zonas)

    # Restricciones (Reglas de Oro):
    # 1. Grúa Torre: Solo puede ubicarse en la Zona_Estable
    problem.addConstraint(lambda g: g == "zona_Estable", ["GrúaTorre"])

    # 2. Excavadora: No puede ingresar a la Zona_Estrecha
    problem.addConstraint(lambda e: e != "zona_Estrecha", ["Excavadora"])

    # 3. Hormigonera: No puede estar en la misma zona que la Grúa Torre
    problem.addConstraint(lambda h, g: h != g, ["Hormigonera", "GrúaTorre"])

    # 4. Exclusividad: Cada zona solo puede albergar una máquina a la vez
    problem.addConstraint(AllDifferentConstraint(), maquinas)

    # Resolución
    soluciones = problem.getSolutions()

    print(f"Se encontraron {len(soluciones)} configuración(es) segura(s):")
    return soluciones
planificar_maquinaria()




Se encontraron 1 configuración(es) segura(s):


[{'GrúaTorre': 'zona_Estable',
  'Hormigonera': 'zona_Estrecha',
  'Excavadora': 'zona_Espera'}]

In [ ]:
import pandas as pd
import numpy as np

soluciones = planificar_maquinaria()
# Convertimos la lista de diccionarios a un DataFrame de Pandas
df = pd.DataFrame(soluciones)


print("Configuraciones seguras encontradas:")
display(df) # Muestra la tabla con el índice numérico

Se encontraron 1 configuración(es) segura(s):
Configuraciones seguras encontradas:


,GrúaTorre,Hormigonera,Excavadora
0,zona_Estable,zona_Estrecha,zona_Espera


## Parámetros de Entrega y Evaluación
El entregable debe cumplir con lo siguiente:

* **Justificación Funcional:** Debes explicar en celdas de texto por qué el modelo de grafos y árboles es superior a una simple lista de if/else para este problema.
* **Documentación:** El código debe estar respaldado por una explicación de cómo opera el motor de inferencia en cada caso.

*Tener en cuenta que se evalua proceso y no resultado. Acordarse de justificar las elecciones en cada caso.*

Un modelo de grafos y árboles es superior a un lista de if/else porque se acorta la ruta a la respuesta de los solicitado.
Mientras que una  lista de condiciones if/else debe evaluar cada condicion secuencilamente hasta encontrar la coincidecia, los grafos y arboles llegan a la solución navegando a través de nodos o ramas especificas sin evaluar opciones irrelevantes a traves de la poda de ramas.

